# 3a. Interventions, seeds 1 to 8

Ten configurations on the same seeds, so every comparison is paired (split over two notebooks to fit Kaggle's session limit):

| Test | Configuration | Compared with |
|---|---|---|
| H1 | queue-actuated adaptive signals | baseline |
| H2 | congestion pricing: toll = 0.2 x each link's untolled delay (from step 2) | baseline |
| H3 | both | baseline |
| H4 | tidal lanes on morning-biased demand | same demand, no tidal lanes |
| H5 | parking enforcement proxy: free-flow speeds x35/30 | baseline |
| H6 | staggered departures: same demand over a 50% longer window | baseline |
| H7 | trips into the 10 core zones x0.75 | baseline, and a uniform cut to the same total |

Horizon 4 hours. Main measure: mean time in system over all vehicles.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("data/jica_net/od_matrix.csv")
tolls = glob.glob("/kaggle/input/**/results/tolls_jica.csv", recursive=True)[0]
os.makedirs("results", exist_ok=True); shutil.copy(tolls, "results/tolls_jica.csv")
if os.path.exists("results/jica_h1h7.partial.csv"): os.remove("results/jica_h1h7.partial.csv")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/compare_jica.py", "--first-seed", "1", "--seeds", "8", "--jobs", "4", tries=3)
shutil.copy("results/jica_h1h7.partial.csv", "/kaggle/working/interventions_seeds1-8.csv")